In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

Try the Eliassen Palm Flux computation

In [ ]:
ds_T=xr.open_dataset('climca/data/ERA5/daily/T500/E5pl00_1D_1990-01_130.nc')

In [ ]:
##we need daily u,v & T data (from ERA5?)

ds_u=xr.open_mfdataset('climca/data/ERA5/daily/u')
display(ds_u)


In [ ]:
def compute_ep_flux(ds):
    """Compute Eliassen-Palm Flux components from ERA5 data.

    Parameters:
    -----------
    ds : xarray.Dataset
        Contains variables: 'u' (m/s), 'v' (m/s), 't' (K)
        Dimensions expected: (time, level, latitude, longitude)
        'level' must be in hPa or Pa.
    """
    # Constants
    a = 6.371e6  # Earth radius (m)
    omega = 7.2921e-5  # Earth rotation rate (rad/s)
    g = 9.80665  # Gravity (m/s^2)
    Rd = 287.05  # Gas constant for dry air (J/kg/K)
    Cp = 1004.0  # Specific heat at constant pressure (J/kg/K)

    # 1. Coordinate Setup
    # Ensure pressure is in Pascals for vertical derivatives
    if ds.level.max() < 2000:
        p = ds.level * 100.0  # Convert hPa to Pa
    else:
        p = ds.level
    lat_rad = np.deg2rad(ds.latitude)

    # 2. Compute Potential Temperature (Theta)
    p_0 = 100000.0  # Reference pressure in Pa
    theta = ds.t * (p_0 / p) ** (Rd / Cp)

    # 3. Calculate Zonal Means and Eddy Deviations
    u_mean = ds.u.mean(dim="longitude")
    v_mean = ds.v.mean(dim="longitude")
    theta_mean = theta.mean(dim="longitude")

    u_prime = ds.u - u_mean
    v_prime = ds.v - v_mean
    theta_prime = theta - theta_mean

    # 4. Compute Eddy Covariances (Zonal Averages)
    uv_prime_mean = (u_prime * v_prime).mean(dim="longitude")
    vtheta_prime_mean = (v_prime * theta_prime).mean(dim="longitude")

    # 5. Geometrical and Coriolis Parameters
    cos_lat = np.cos(lat_rad)
    sin_lat = np.sin(lat_rad)
    f = 2 * omega * sin_lat

    # 6. Vertical Derivative of Mean Theta (dTh/dp)
    # differentiated along the 'level' axis
    dtheta_dp = theta_mean.gradient({"level": p.values}, edge_order=2)[
        "level"
    ]

    # 7. Compute EP Flux Components
    # F_phi (Meridional component)
    F_phi = -a * cos_lat * uv_prime_mean

    # F_p (Vertical component)
    # Avoid division by zero at the equator where f=0
    F_p = f * a * cos_lat * (vtheta_prime_mean / dtheta_dp)

    # 8. Compute Divergence
    # Meridional derivative: d/dphi(F_phi * cos_lat)
    F_phi_cos = F_phi * cos_lat
    dF_phi_dlat = F_phi_cos.gradient({"latitude": lat_rad.values}, edge_order=2)[
        "latitude"
    ]
    div_phi = (1.0 / (a * cos_lat)) * dF_phi_dlat

    # Vertical derivative: d(F_p)/dp
    div_p = F_p.gradient({"level": p.values}, edge_order=2)["level"]

    ep_div = div_phi + div_p

    # Package into a clean Dataset
    ep_ds = xr.Dataset(
        data_vars={
            "F_phi": F_phi,
            "F_p": F_p,
            "EP_div": ep_div,
        }
    )

    return ep_ds


# --- Usage Example ---
# ds = xr.open_dataset("era5_daily_uvt.nc")
# ep_results = compute_ep_flux(ds)
